In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ibge.gold

In [0]:
# Import das funções
from pyspark.sql.functions import col, sum as sum_
from functools import reduce
import re

ANO = 2022

In [0]:
# Sanitiza os nomes gerados pelo pivot
def sanitize_column_name(name):
    replacements = {
        'ã': 'a', 'â': 'a', 'á': 'a', 'à': 'a',
        'ê': 'e', 'é': 'e', 'è': 'e',
        'í': 'i', 'î': 'i',
        'ô': 'o', 'õ': 'o', 'ó': 'o',
        'ú': 'u', 'û': 'u',
        'ç': 'c'
    }
    name = name.lower()
    for k, v in replacements.items():
        name = name.replace(k, v)
    name = re.sub(r"[ ,;{}()\n\t=]", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

ANO = 2022

# PIB
pib_df = (
    spark.read.table("ibge.silver.pib")
    .filter(col("ano") == ANO)
    .groupBy("cod_municipio", "municipio")
    .agg(sum_("valor").alias("pib_total"))
)

# POPULAÇÃO TOTAL
pop_df = (
    spark.read.table("ibge.silver.pop_area")
    .filter(col("ano") == ANO)
    .groupBy("cod_municipio", "municipio")
    .agg(sum_("valor").alias("pop_total"))
)

# ALFABETIZAÇÃO
alfabetizacao_df = (
    spark.read.table("ibge.silver.alfabetizacao")
    .filter(col("ano") == ANO)
    .filter(col("sexo") == "Total")
    .groupBy("cod_municipio", "municipio")
    .agg(sum_("valor").alias("taxa_alfabetizacao"))
)

# EMPREGO
emprego_df = (
    spark.read.table("ibge.silver.emprego")
    .filter(col("ano") == ANO)
    .groupBy("cod_municipio", "municipio")
    .agg(sum_("valor").alias("emprego_total"))
)

# ÁGUA
agua_df = (
    spark.read.table("ibge.silver.agua")
    .filter(col("ano") == ANO)
    .groupBy("cod_municipio", "municipio")
    .agg(sum_("valor").alias("agua_atendida"))
)

# ESGOTO
esgoto_df = (
    spark.read.table("ibge.silver.esgoto")
    .filter(col("ano") == ANO)
    .groupBy("cod_municipio", "municipio")
    .agg(sum_("valor").alias("esgoto_atendido"))
)

# INSTRUÇÃO (pivot)
instrucao_df = (
    spark.read.table("ibge.silver.instrucao")
    .filter(col("ano") == ANO)
    .filter(col("sexo") == "Total")
    .filter(col("cor_raca") == "Total")
    .filter(col("grupo_idade") == "Total")
    .groupBy("cod_municipio", "municipio")
    .pivot("nivel_instrucao")
    .agg(sum_("valor"))
)

# Sanitiza os nomes gerados pelo pivot
for old_name in instrucao_df.columns:
    new_name = sanitize_column_name(old_name)
    if old_name != new_name:
        instrucao_df = instrucao_df.withColumnRenamed(old_name, new_name)

# JOIN DE TUDO
dfs = [
    pib_df,
    pop_df,
    alfabetizacao_df,
    emprego_df,
    agua_df,
    esgoto_df,
    instrucao_df
]

gold_df = reduce(
    lambda df1, df2: df1.join(df2, ["cod_municipio", "municipio"], "left"),
    dfs
)

# SALVAR GOLD
(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ibge.gold.indicadores_municipio")
)